In [ ]:
import json
import sys
import re
from rich import print as rp
from collections import Counter
from pathlib import Path
from datetime import datetime

nb_dir = Path.cwd()
project_root = nb_dir.parent.parent
sys.path.insert(0, str(project_root))

from scripts.text_matching import normalise_text

In [ ]:
db_people_file = Path(project_root / "data_reload/db_exports/people_variants_2026-08-11.json")


new_people_file = Path(project_root / "data_reload/reparse_missing/prepping/missing_reparsed_people.json")

with open(db_people_file, "r") as f:
   db_people = json.load(f)

with open(new_people_file, "r") as f:
   new_people = json.load(f)


In [ ]:
# let's find problematic uids first
uid_single = []
uid_2 = []
uid_3 = []
uid_multiple = []
uid_2_fix = []

counts = Counter()

for person in db_people:
    uid = person["unified_id"]
    last = person["family_name"]
    first = person["given_names"]
    single = person["single_name"]
    uid_parts = uid.split("_")
    counts["people"] += 1

    if len(uid_parts) <= 1:
        # uid_single.append({"uid_parts": uid_parts, **person})
        # counts["single"] += 1
        if last or first and not single:
            uid_2_fix.append({"uid_parts": uid_parts, **person})
            counts["needs fixing"] += 1
        elif not single:
            uid_2_fix.append({"uid_parts": uid_parts, **person})
            counts["needs fixing"] += 1
        else:
            uid_single.append({"uid_parts": uid_parts, "is_single": True, **person})
    elif len(uid_parts) == 2:
        uid_2.append({"uid_parts": uid_parts, **person})
        counts["two"] += 1
    elif len(uid_parts) == 3:
        uid_3.append({"uid_parts": uid_parts, **person})
        counts["three"] += 1
        if last and "-" in last:
            uid_2_fix.append({"uid_parts": uid_parts, **person})
            counts["needs fixing"] += 1
    else:
        uid_multiple.append({"uid_parts": uid_parts, **person})
        counts["multi"] += 1
        if last and "-" in last:
            uid_2_fix.append({"uid_parts": uid_parts, **person})
            counts["needs fixing"] += 1

# rp(counts)
# # rp(uid_2_fix)
# rp(uid_single)
# with open("uids_2_fix.json", "w") as f:
#     json.dump(uid_2_fix, f, ensure_ascii=False, indent=2)

# with open("singles_check.json", "w") as f:
#     json.dump(uid_single, f, ensure_ascii=False, indent=2)


In [23]:
with open("singles_check.json", "r") as f:
   checked_singles = json.load(f)

real_singles_dict = {}
false_singles_dict = {}

for entry in checked_singles:
    person_id = entry["person_id"]
    uid = entry["unified_id"]
    if entry["is_single"] == True:
        real_singles_dict[person_id] = entry
    else:
        false_singles_dict[person_id] = entry


rp(len(real_singles_dict))
rp(len(false_singles_dict))
with open("false_singles.json", "w") as f:
    json.dump(false_singles_dict, f, ensure_ascii=False, indent=2)


97

35